<a href="https://colab.research.google.com/github/SaiSiddharthD/Deepfake-Detection-Optimization/blob/main/Spectrograms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import librosa, librosa.display
import numpy as np
import tqdm.notebook as tq
import os

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### File Setting

In [3]:
!unzip -o -q /content/drive/MyDrive/LibriSeVoc_Lite.zip
os.chdir("LibriSeVoc Lite/Real")
files = sorted(os.listdir())
SpecsReal = []
MfccReal = []

for file in tq.tqdm(files):
    if file.endswith(".wav"):
        signal, sr = librosa.load(file)
        spec = librosa.amplitude_to_db(np.abs(librosa.stft(signal)))
        SpecsReal.append(spec)
        mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20)
        MfccReal.append(mfcc)
os.chdir("../Fake/diffwave")
files = sorted(os.listdir())
SpecsDiffwave = []
MfccDiffwave = []

for file in tq.tqdm(files):
    if file.endswith(".wav"):
        signal, sr = librosa.load(file)
        spec = librosa.amplitude_to_db(np.abs(librosa.stft(signal)))
        SpecsDiffwave.append(spec)
        mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20)
        MfccDiffwave.append(mfcc)
os.chdir("../melgan")
files = sorted(os.listdir())
SpecsMelgan = []
MfccMelgan = []

for file in tq.tqdm(files):
    if file.endswith(".wav"):
        signal, sr = librosa.load(file)
        spec = librosa.amplitude_to_db(np.abs(librosa.stft(signal)))
        SpecsMelgan.append(spec)
        mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20)
        MfccMelgan.append(mfcc)
os.chdir("../parallel_wave_gan")
files = sorted(os.listdir())
SpecsParallelWaveGan = []
MfccParallelWaveGan = []

for file in tq.tqdm(files):
    if file.endswith(".wav"):
        signal, sr = librosa.load(file)
        spec = librosa.amplitude_to_db(np.abs(librosa.stft(signal)))
        SpecsParallelWaveGan.append(spec)
        mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20)
        MfccParallelWaveGan.append(mfcc)
os.chdir("../wavegrad")
files = sorted(os.listdir())
SpecsWavegrad = []
MfccWavegrad = []

for file in tq.tqdm(files):
    if file.endswith(".wav"):
        signal, sr = librosa.load(file)
        spec = librosa.amplitude_to_db(np.abs(librosa.stft(signal)))
        SpecsWavegrad.append(spec)
        mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20)
        MfccWavegrad.append(mfcc)
os.chdir("../wavernn")
files = sorted(os.listdir())
SpecsWavernn = []
MfccWavernn = []

for file in tq.tqdm(files):
    if file.endswith(".wav"):
        signal, sr = librosa.load(file)
        spec = librosa.amplitude_to_db(np.abs(librosa.stft(signal)))
        SpecsWavernn.append(spec)
        mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20)
        MfccWavernn.append(mfcc)
os.chdir("../wavenet")
files = sorted(os.listdir())
SpecsWavenet = []
MfccWavenet = []

for file in tq.tqdm(files):
    if file.endswith(".wav"):
        signal, sr = librosa.load(file)
        spec = librosa.amplitude_to_db(np.abs(librosa.stft(signal)))
        SpecsWavenet.append(spec)
        mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20)
        MfccWavenet.append(mfcc)

TotalSpecs=SpecsReal+SpecsDiffwave+SpecsMelgan+SpecsParallelWaveGan+SpecsWavegrad+SpecsWavernn+SpecsWavenet

  0%|          | 0/253 [00:00<?, ?it/s]

  0%|          | 0/252 [00:00<?, ?it/s]

  0%|          | 0/252 [00:00<?, ?it/s]

  0%|          | 0/252 [00:00<?, ?it/s]

  0%|          | 0/252 [00:00<?, ?it/s]

  0%|          | 0/252 [00:00<?, ?it/s]

  0%|          | 0/253 [00:00<?, ?it/s]

### Splitting The Data

In [4]:
shortest=min([x.shape[1] for x in TotalSpecs])

In [5]:
import matplotlib.pyplot as plt
import os
import numpy as np
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Flatten, Dense, MaxPool2D, GlobalAveragePooling2D
from sklearn.model_selection import train_test_split
import gdown
from ipywidgets import interact, IntSlider
import pandas as pd
import datetime
%load_ext tensorboard
!rm -rf ./logs/
from tensorflow.keras.utils import to_categorical
np.random.seed(12)
tf.random.set_seed(12)
DF_PATH = ('https://dpl6hyzg28thp.cloudfront.net/media/metadata.csv')
IMG_PATH = ('/content/drive/MyDrive/Spectrogram_Images')
NUM_IMS = 500
CHECKPOINT_PATH = 'checkpoint'
INPUT_SHAPE = (1025, shortest,1)
!rm -rf logs
!mkdir logs
!mkdir logs/fit/

def load_image_from_name(name):
    path = Path(IMG_PATH) / f'{name}.jpg'
    return plt.imread(path) / 255

def plot_dist(df):
    num2labs = {
        0:"diffwave"
        , 1:"real"
    }
    counts = df["level"].map(num2labs).value_counts() / len(df)
    counts.plot.bar()
    plt.title("Percentage of images in each class")
    plt.show()

def plot_helper(df, index):
    row = df.iloc[index]
    img = load_image_from_name(row.image)
    plt.title(f"Level: {row.level}")
    plt.imshow(img)

def normalize(img):
    return (img - img.min()) / (img.max() - img.min())

def MobileNet(freeze=False):
    model = tf.keras.applications.MobileNet(input_shape=INPUT_SHAPE, weights='imagenet', include_top=False)
    model.trainable= not freeze
    return tf.keras.Sequential([
        model,
        GlobalAveragePooling2D(),
        Dense(1, activation='sigmoid')
    ])

In [6]:
y_train= ["diffwave"]*201 + ['melgan']*201 + ['parallel_wave_gan']*201 + ["real"]*201 + ['wavegrad']*201 + ['wavenet']*201 + ['wavernn']*201
y_val=["diffwave"]*51 + ['melgan']*51 + ['parallel_wave_gan']*51 + ["real"]*51 + ['wavegrad']*51 + ['wavenet']*51 + ['wavernn']*51
IDX=list(range(252))
x_train, x_val = train_test_split(IDX, shuffle=True, test_size=0.2)

SpecsTrain = []
SpecsTest = []
MfccTrain = []
MfccTest = []


for spec_vocoder, mfcc_vocoder in zip(
    [SpecsDiffwave, SpecsMelgan, SpecsParallelWaveGan, SpecsReal, SpecsWavegrad, SpecsWavenet, SpecsWavernn],
    [MfccDiffwave, MfccMelgan, MfccParallelWaveGan, MfccReal, MfccWavegrad, MfccWavenet, MfccWavernn]
):

  spec_vocoder = np.array([x[:,:shortest] for x in spec_vocoder])
  SpecsTrain.extend(spec_vocoder[x_train,:,:])
  SpecsTest.extend(spec_vocoder[x_val,:,:])


  mfcc_vocoder = np.array([x[:,:shortest] for x in mfcc_vocoder])
  MfccTrain.extend(mfcc_vocoder[x_train,:,:])
  MfccTest.extend(mfcc_vocoder[x_val,:,:])

SpecsTrain = np.array(SpecsTrain)
SpecsTest = np.array(SpecsTest)
MfccTrain = np.array(MfccTrain)
MfccTest = np.array(MfccTest)

In [7]:
y_train = to_categorical(pd.Series(y_train).astype("category").cat.codes, 7)
y_val = to_categorical(pd.Series(y_val).astype("category").cat.codes, 7)

###  Training Preparation


In [8]:
import math
from tensorflow.keras.callbacks import LearningRateScheduler
import datetime
import tensorflow as tf

def lr_schedule(epoch, lr):
    if epoch < 10:
        return 0.0001
    else:
        return float(lr * math.exp(-0.05))

def train(model, x_spec, x_mfcc, y, x_val_spec, x_val_mfcc, y_val, epochs):
    log_dir = "logs/" + model.name + "/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
    lr_callback = LearningRateScheduler(lr_schedule)

    optimizer = tf.optimizers.Adam()
    model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics=["accuracy"])

    return model.fit(
        [x_spec, x_mfcc],
        y,
        epochs=epochs,
        validation_data=([x_val_spec, x_val_mfcc], y_val),
        callbacks=[tensorboard_callback, lr_callback],
        verbose=1,
    )

### Creating our Models



In [9]:
from tensorflow.keras.layers import Input, Concatenate, Dropout, BatchNormalization, Dense, Flatten, Conv2D, MaxPool2D
from tensorflow.keras.models import Model

input_spec = Input(shape=INPUT_SHAPE, name="spectrogram_input")
x = Conv2D(32, 3, activation="relu")(input_spec)
x = MaxPool2D()(x)
x = Conv2D(64, 3, activation="relu")(x)
x = MaxPool2D()(x)
x = Conv2D(64, 3, activation="relu")(x)
x = Flatten()(x)

input_mfcc = Input(shape=(20, shortest, 1), name="mfcc_input")
y = Flatten()(input_mfcc)
y = BatchNormalization()(y)
y = Dense(64, activation="relu")(y)

fused = Concatenate()([x, y])
fused = Dense(32, activation="relu")(fused)
fused = Dropout(0.5)(fused)

outputs = Dense(7, activation='softmax')(fused)

conv_model = Model(inputs=[input_spec, input_mfcc], outputs=outputs, name="hybrid_late_fusion_v2")

### Model Training

In [ ]:
hist = train(conv_model, SpecsTrain, MfccTrain, y_train, SpecsTest, MfccTest, y_val, epochs=32)

Epoch 1/32
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 539ms/step - accuracy: 0.1152 - loss: 34.1535

### Plotting

In [ ]:
import seaborn as sns
import matplotlib.pyplot as pyplot
from keras.callbacks import ReduceLROnPlateau
def plot_acc(history, ax = None, xlabel = 'Epoch #'):
    history = history.history
    history.update({'epoch':list(range(len(history['val_accuracy'])))})
    history = pd.DataFrame.from_dict(history)

    best_epoch = history.sort_values(by = 'val_accuracy', ascending = False).iloc[0]['epoch']

    if not ax:
      f, ax = plt.subplots(1,1)
    sns.lineplot(x = 'epoch', y = 'val_accuracy', data = history, label = 'Validation', ax = ax)
    sns.lineplot(x = 'epoch', y = 'accuracy', data = history, label = 'Training', ax = ax)
    ax.axhline(0.1, linestyle = '--',color='red', label = 'Chance')
    ax.axvline(x = best_epoch, linestyle = '--', color = 'green', label = 'Best Epoch')
    ax.legend(loc = 7)
    ax.set_ylim([min(min(history['val_accuracy']), min(history['accuracy'])), max(max(history['accuracy']), max(history['val_accuracy']))])

    ax.set_xlabel(xlabel)
    ax.set_ylabel('Accuracy (Fraction)')

    plt.show()

learning_rate_reduction = ReduceLROnPlateau(monitor='val_acc',
                                            patience=3,
                                            verbose=1,
                                            factor=0.5,
                                            min_lr=0.00001)

def plot_activations(model, input_image, im_per_row=16):
  layer_outputs = [layer.output for layer in model.layers[:8]]
  activation_model = models.Model(inputs=model.inputs, outputs=layer_outputs)
  activations = activation_model.predict(test_im.reshape(1,28,28,1))
  layer_names = []
  for layer in model.layers[:-1]:
      layer_names.append(layer.name)
  images_per_row = im_per_row
  for layer_name, layer_activation in zip(layer_names, activations):
      if layer_name.startswith('conv') or layer_name.startswith('max'):
          n_features = layer_activation.shape[-1]
          size = layer_activation.shape[1]
          n_cols = n_features // images_per_row
          display_grid = np.zeros((size * n_cols, images_per_row * size))
          for col in range(n_cols):
              for row in range(images_per_row):
                  channel_image = layer_activation[0,:, :, col * images_per_row + row]
                  channel_image -= channel_image.mean()
                  channel_image /= channel_image.std()
                  channel_image *= 64
                  channel_image += 128
                  channel_image = np.clip(channel_image, 0, 255).astype('uint8')
                  display_grid[col * size : (col + 1) * size,
                              row * size : (row + 1) * size] = channel_image
          scale = 1. / size
          plt.figure(figsize=(scale * display_grid.shape[1],
                              scale * display_grid.shape[0]))
          plt.title(layer_name)
          plt.grid(False)
          plt.imshow(display_grid, aspect='auto', cmap='viridis')

def plot_kernels(model, img_per_row=16):
  filters, biases = model.layers[0].get_weights()
  # normalize filter values to 0-1 so we can visualize them
  f_min, f_max = filters.min(), filters.max()
  filters = (filters - f_min) / (f_max - f_min)
  n_filters = filters.shape[-1]
  fig, ax = pyplot.subplots(n_filters//img_per_row, img_per_row, figsize=(15,2))
  for i in range(n_filters):
    f = filters[:, :, :, i]
    ax[i//img_per_row, i%img_per_row].set_xticks([])
    ax[i//img_per_row, i%img_per_row].set_yticks([])
    ax[i//img_per_row, i%img_per_row].imshow(f.reshape((5,5)), cmap='gray')
  pyplot.show()

In [ ]:
plot_acc(hist)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
y_true=np.argmax(y_val, axis=1)
y_pred = np.argmax(conv_model.predict(SpecsTest), axis=1)

In [ ]:
class_names=['F1', 'F2', 'F3', 'R', 'F4', 'F5', 'F6']

In [ ]:
cm = confusion_matrix(y_true, y_pred)
disp=ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Reds');

In [ ]:
RealPrecision=34/(3+1+34+2+1)

##TensorBoard

As we tune our hyperparameters, we'll have a **lot** of data to keep track of. Fortunately, there's a nifty tool to help us explore our model and results: TensorBoard. Run the line below to start it up:

In [ ]:
%tensorboard --logdir logs